# Synthetic Distribution model Building using SHIFT
1. Install dependencies for dashboard

1. import modules to create an interactive map

In [2]:
coordinates = [
    [-76.682145, 40.267476],
    [-76.679848, 40.26782],
    [-76.679913, 40.268229],
    [-76.680256, 40.26854],
    [-76.681029, 40.268655],
    [-76.680943, 40.269081],
    [-76.680342, 40.269114],
    [-76.679333, 40.267886],
    [-76.674054, 40.268328],
    [-76.670341, 40.268802],
    [-76.666521, 40.269637],
    [-76.663108, 40.27098],
    [-76.659825, 40.272077],
    [-76.655855, 40.273207],
    [-76.65506, 40.271766],
    [-76.651348, 40.269588],
    [-76.651004, 40.270391],
    [-76.65006, 40.270849],
    [-76.649759, 40.270309],
    [-76.650596, 40.268917],
    [-76.655254, 40.271487],
    [-76.655897, 40.272879],
    [-76.658966, 40.272093],
    [-76.657829, 40.269916],
    [-76.657099, 40.269425],
    [-76.656799, 40.268753],
    [-76.657489, 40.265954],
    [-76.666781, 40.2643],
    [-76.66646, 40.263039],
    [-76.666781, 40.262269],
    [-76.668928, 40.262188],
    [-76.67116, 40.262712],
    [-76.673714, 40.261844],
    [-76.672984, 40.26109],
    [-76.673928, 40.260534],
    [-76.675516, 40.261058],
    [-76.680646, 40.259813],
    [-76.683779, 40.259437],
    [-76.684831, 40.258798],
    [-76.685646, 40.259781],
    [-76.688844, 40.259748],
    [-76.688822, 40.260665],
    [-76.690024, 40.261909],
    [-76.688715, 40.262433],
    [-76.686526, 40.262793],
    [-76.685732, 40.262843],
    [-76.685474, 40.265397],
    [-76.683948, 40.266527],
]

SyntaxError: invalid syntax (3091362153.py, line 2)

In [3]:
from shift_modeling import get_parcels_in_polygon

plot_manager, parcels, polygon_center = get_parcels_in_polygon(coordinates)
plot_manager.show()

## 3. Query Parcels Inside the Polygon
Call the helper to fetch parcel geometries inside the selected polygon and compute a polygon center for plotting.

Outputs from this step:
- `plot_manager`: map object for visualization
- `parcels`: list of parcel models used for graph creation
- `polygon_center`: map center used in subsequent plots

4. Build graph for the queried parcels

In [ ]:
from shift_modeling import build_graph_from_parcels

parcels_per_cluster = 2
graph = build_graph_from_parcels(parcels, parcels_per_cluster, source_location=polygon_center)

2025-01-22 16:52:26.863 | INFO     | shift.openstreet_roads:get_road_network:43 - Attempting to fecth road network for POLYGON ((-76.68771040426707 40.260275919423684, -76.65889975423941 40.260275919423684, -76.65889975423941 40.27090628352561, -76.68771040426707 40.27090628352561, -76.68771040426707 40.260275919423684))
2025-01-22 16:52:27.574 | INFO     | shift.graph.openstreet_graph_builder:get_distribution_graph:309 - Building secondary for GeoLocation(longitude=-76.67243350389803, latitude=40.265529035994824): 7faa8d2e-f879-4548-a524-803bdf03cfbf
2025-01-22 16:52:27.577 | INFO     | shift.graph.openstreet_graph_builder:get_distribution_graph:309 - Building secondary for GeoLocation(longitude=-76.6813625276618, latitude=40.264349352325794): ead071e5-0246-4fe0-ac18-cac4e0b46122
2025-01-22 16:52:27.581 | INFO     | shift.graph.openstreet_graph_builder:get_distribution_graph:309 - Building secondary for GeoLocation(longitude=-76.65907970907075, latitude=40.27072632869427): eab3a7ff-16

5. Visualising the graph

In [ ]:
from shift_modeling import plot_graph

graph_plot_manager = plot_graph(graph, polygon_center)
graph_plot_manager.show()

6. Map phases to graph edges

In [ ]:
from shift import TransformerPhaseMapperModel, TransformerTypes, BalancedPhaseMapper
from shift import add_phase_mapper_to_plot

from gdm.quantities import PositiveApparentPower
from gdm import DistributionTransformer

mapper = [
    TransformerPhaseMapperModel(
        tr_name=el.name,
        tr_type=TransformerTypes.THREE_PHASE,
        tr_capacity=PositiveApparentPower(
            500,
            "kilova",
        ),
        location=graph.get_node(from_node).location,
    )
    for from_node, _, el in graph.get_edges()
    if el.edge_type is DistributionTransformer
]

phase_mapper = BalancedPhaseMapper(graph, method="greedy", mapper=mapper)
phase_mapper.node_phase_mapping
phase_mapper.transformer_phase_mapping

add_phase_mapper_to_plot(phase_mapper, graph_plot_manager)
graph_plot_manager.show()

7. Map voltage levels to the graph

In [ ]:
from shift import TransformerVoltageMapper, TransformerVoltageModel
from shift import add_voltage_mapper_to_plot

from gdm.quantities import PositiveVoltage
from gdm import DistributionTransformer

voltage_mapper = TransformerVoltageMapper(
    graph,
    xfmr_voltage=[
        TransformerVoltageModel(
            name=el.name,
            voltages=[PositiveVoltage(12.47, "kilovolt"), PositiveVoltage(480, "volt")],
        )
        for _, _, el in graph.get_edges()
        if el.edge_type is DistributionTransformer
    ],
)
voltage_mapper.node_voltage_mapping
voltage_mapper.xfmr_voltage
add_voltage_mapper_to_plot(voltage_mapper, graph_plot_manager)
graph_plot_manager.show()

8. Load existing catalog

In [2]:
from gdm import DistributionSystem

model_path = r"C:\Users\alatif\Desktop\Jupyter Notebooks\SDC model build\model\p3uhs2_1247.json"
catalog = DistributionSystem.from_json(model_path)
catalog.info()

ModuleNotFoundError: No module named 'gdm'

In [ ]:
from shift_modeling import CustomLoadMapper

load_mapper = CustomLoadMapper(
    graph=graph,
    catalog_sys=catalog,
    voltage_mapper=voltage_mapper,
    phase_mapper=phase_mapper,
    parcels=parcels,
    zip_code="08618",
)
print(load_mapper)

Using existing metadata file
Using existing metadata file


In [ ]:
from shift import DistributionSystemBuilder

builder = DistributionSystemBuilder(
    name="Test system",
    dist_graph=graph,
    phase_mapper=phase_mapper,
    voltage_mapper=voltage_mapper,
    equipment_mapper=load_mapper,
)
sys = builder.get_system()

[GeoLocation(longitude=-76.6724732, latitude=40.2657259), GeoLocation(longitude=-76.672239, latitude=40.2653962), GeoLocation(longitude=-76.6723938, latitude=40.2653322), GeoLocation(longitude=-76.672628, latitude=40.2656618), GeoLocation(longitude=-76.6724732, latitude=40.2657259)]
exact_zip={'zip_code': '08618', 'zip_code_type': 'STANDARD', 'active': True, 'city': 'Trenton', 'acceptable_cities': ['Ewing'], 'unacceptable_cities': ['Ewing Township', 'Ewing Twp'], 'state': 'NJ', 'county': 'Mercer County', 'timezone': 'America/New_York', 'area_codes': ['609'], 'world_region': 'NA', 'country': 'US', 'lat': '40.2544', 'long': '-74.7876'}


C:\Users\alatif\Desktop\Jupyter Notebooks\SDC model build\resstock_comstock_interface.py:182: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\alatif\Desktop\Jupyter Notebooks\SDC model build\resstock_comstock_interface.py:182: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



[GeoLocation(longitude=-76.6816485, latitude=40.2639546), GeoLocation(longitude=-76.6816279, latitude=40.2642723), GeoLocation(longitude=-76.6815002, latitude=40.2642669), GeoLocation(longitude=-76.6814879, latitude=40.2645029), GeoLocation(longitude=-76.6815992, latitude=40.2645062), GeoLocation(longitude=-76.6815984, latitude=40.2645223), GeoLocation(longitude=-76.6817767, latitude=40.2645278), GeoLocation(longitude=-76.6817729, latitude=40.264601), GeoLocation(longitude=-76.6818602, latitude=40.2646037), GeoLocation(longitude=-76.681862, latitude=40.2645708), GeoLocation(longitude=-76.6820137, latitude=40.2645754), GeoLocation(longitude=-76.682016, latitude=40.2645323), GeoLocation(longitude=-76.6820706, latitude=40.264534), GeoLocation(longitude=-76.6820722, latitude=40.2643016), GeoLocation(longitude=-76.682035, latitude=40.2643002), GeoLocation(longitude=-76.6820333, latitude=40.2642874), GeoLocation(longitude=-76.6818104, latitude=40.2642796), GeoLocation(longitude=-76.6818321, 

C:\Users\alatif\Desktop\Jupyter Notebooks\SDC model build\resstock_comstock_interface.py:182: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



AWS path: nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2021/comstock_amy2018_release_1/timeseries_individual_buildings/by_county/upgrade=0/county=G3400210/161183-0.parquet
[GeoLocation(longitude=-76.6812552, latitude=40.2644953), GeoLocation(longitude=-76.681219, latitude=40.2644943), GeoLocation(longitude=-76.6811009, latitude=40.2644912), GeoLocation(longitude=-76.6811063, latitude=40.2644288), GeoLocation(longitude=-76.6809172, latitude=40.2644221), GeoLocation(longitude=-76.6809145, latitude=40.2644963), GeoLocation(longitude=-76.6808622, latitude=40.2644973), GeoLocation(longitude=-76.6808314, latitude=40.2645188), GeoLocation(longitude=-76.680881, latitude=40.2645254), GeoLocation(longitude=-76.6808796, latitude=40.2645741), GeoLocation(longitude=-76.6810929, latitude=40.2647573), GeoLocation(longitude=-76.6812525, latitude=40.2646574), GeoLocation(longitude=-76.6812552, latitude=40.2644953)]
exact_zip={'zip_code': '08618', 'zip_code_type': 'STANDARD', 'act

...

Component Information                       
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Type                             ┃ Count ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ DistributionBus                  │   231 │
│ DistributionLoad                 │    56 │
│ DistributionTransformer          │    31 │
│ DistributionTransformerEquipment │     2 │
│ DistributionVoltageSource        │     1 │
│ LoadEquipment                    │    56 │
│ Location                         │   200 │
│ MatrixImpedanceBranch            │   199 │
│ MatrixImpedanceBranchEquipment   │     3 │
│ PhaseLoadEquipment               │   168 │
│ PhaseVoltageSourceEquipment      │     3 │
│ ThermalLimitSet                  │     1 │
│ VoltageLimitSet                  │   462 │
│ VoltageSourceEquipment           │     1 │
│ WindingEquipment                 │     4 │
└──────────────────────────────────┴───────┘

Time Series Summary                                                                                                
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━
┃                      ┃                  ┃                     ┃            ┃                ┃     No. Components 
┃ Component Type       ┃ Time Series Type ┃        Initial time ┃ Resolution ┃ No. Components ┃   with Time Series 
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━
│ DistributionLoad     │ SingleTimeSeries │ 2018-01-01 00:00:00 │    1:00:00 │             56 │                 56 
└──────────────────────┴──────────────────┴─────────────────────┴────────────┴────────────────┴────────────────────

Graph with 231 nodes and 230 edges
Graph with 231 nodes and 230 edges


C:\Users\alatif\Desktop\Jupyter Notebooks\SDC model build\shift_modeling.py:215: DeprecationWarning:

This function will be deprecated in future version (after v2.0.0)
        You can use `system.get_undirected_graph()` instead.
        



KeyError: <class 'gdm.distribution.components.distribution_vsource.DistributionVoltageSource'>